In [ ]:
import requests
import geopandas as gpd
import json
import pandas as pd
import os

In [ ]:
# damage points (0), damage lines (1), damage polygons (2) 
layer_indices = [0, 1, 2]
base_url = "https://services.dat.noaa.gov/arcgis/rest/services/nws_damageassessmenttoolkit/DamageViewer/MapServer"

# storage for results
layer_data = {}

for num in layer_indices:
    print(f"Fetching layer {num}...")
    url = f"{base_url}/{num}/query"
    
    all_features = []
    offset = 0
    batch_size = 2000

    while True:
        params = {
            "f": "geojson",
            "where": "1=1",
            "outFields": "*",
            "resultRecordCount": batch_size,
            "resultOffset": offset
        }

        response = requests.get(url, params=params)
        data = response.json()

        features = data.get("features", [])
        if not features:
            break

        all_features.extend(features)
        offset += batch_size

    # store in geodataframe
    if all_features:
        gdf = gpd.GeoDataFrame.from_features(all_features)
        gdf.set_crs("EPSG:4326", inplace=True)
        layer_data[f"layer_{num}"] = gdf
        print(f"Layer {num}: {gdf.shape[0]} records")
    else:
        print(f"Layer {num}: No features found")

for name, df in layer_data.items():
    print(f"{name}: {df.shape}")


layer_names = {
    "layer_0": "damage_points",
    "layer_1": "damage_lines",
    "layer_2": "damage_polygons"
}

# Save to one GeoPackage with named layers
for key, df in layer_data.items():
    layer_label = layer_names.get(key, key)

    # Save to combined GeoPackage
    df.to_file("damage_data.gpkg", layer=layer_label, driver="GPKG")
    print(f"Saved layer '{layer_label}' to damage_data.gpkg")

    # Save to separate GPKG file
    df.to_file(f"{layer_label}.gpkg", layer=layer_label, driver="GPKG")
    print(f"Saved '{layer_label}' to its own file: {layer_label}.gpkg")


In [ ]:
damage_points = gpd.read_file('damage_points.gpkg')
damage_points["stormdate"] = pd.to_datetime(damage_points["stormdate"], unit="ms")
damage_points["surveydate"] = pd.to_datetime(damage_points["surveydate"], unit="ms")
damage_points

In [ ]:
"""Slow and downloads a low and higher res img for some reason. Need to figure out how to... not do that."""
# import os
# import requests
# 
# layer_id = 0
# out_dir = "dat_images_full"
# os.makedirs(out_dir, exist_ok=True)
# 
# object_ids = gdf["objectid"].dropna().astype(int).unique()
# 
# for obj_id in object_ids:
#     try:
#         # Step 1: Query for attachments
#         info_url = f"https://services.dat.noaa.gov/arcgis/rest/services/nws_damageassessmenttoolkit/DamageViewer/FeatureServer/{layer_id}/{obj_id}/attachments?f=json"
#         resp = requests.get(info_url)
#         data = resp.json()
#         attachments = data.get("attachmentInfos", [])
# 
#         for att in attachments:
#             att_id = att["id"]
#             filename = att["name"]
# 
#             # Step 2: Download actual image
#             download_url = f"https://services.dat.noaa.gov/arcgis/rest/services/nws_damageassessmenttoolkit/DamageViewer/FeatureServer/{layer_id}/{obj_id}/attachments/{att_id}"
#             out_path = os.path.join(out_dir, f"{obj_id}_{att_id}_{filename}")
# 
#             img = requests.get(download_url)
#             if img.status_code == 200:
#                 with open(out_path, "wb") as f:
#                     f.write(img.content)
#             else:
#                 print(f"Failed: {download_url}")
#     except Exception as e:
#         print(f"Error on objectid {obj_id}: {e}")


In [ ]:
valid_ef = ["EF0", "EF1", "EF2", "EF3", "EF4", "EF5", "EFU", "UNKNOWN", "N/A"]
tornado_points_df = damage_points[damage_points.efscale.isin(valid_ef)]

In [ ]:
tornado_points_df[tornado_points_df.efscale == "UNKNOWN"].comments

In [ ]:
tornado_points_df[tornado_points_df["image"].notnull()].groupby("efscale").size().reset_index(name="image_count")

Get images for higher end tornadoes. Try to retrieve more at your own risk... (many many images and each has to be 2 separate queries)

In [ ]:
# import pandas as pd
# import requests
# from concurrent.futures import ThreadPoolExecutor, as_completed
# 
# # Filter to EF4 and EF5 first
# ef4_5_df = tornado_points_df[tornado_points_df["efscale"].isin(["EF4", "EF5"])]
# object_ids = ef4_5_df["objectid"].dropna().astype(int).unique()
# 
# def fetch_attachments(obj_id):
#     try:
#         url = f"https://services.dat.noaa.gov/arcgis/rest/services/nws_damageassessmenttoolkit/DamageViewer/FeatureServer/0/{obj_id}/attachments?f=json"
#         resp = requests.get(url, timeout=10).json()
#         return [
#             {
#                 "objectid": obj_id,
#                 "attachment_id": att["id"],
#                 "name": att["name"],
#                 "size": att.get("size")
#             }
#             for att in resp.get("attachmentInfos", [])
#         ]
#     except Exception as e:
#         return [{"objectid": obj_id, "error": str(e)}]
# 
# # Run with up to 8 concurrent threads (adjust as needed)
# attachment_records = []
# with ThreadPoolExecutor(max_workers=8) as executor:
#     futures = [executor.submit(fetch_attachments, oid) for oid in object_ids]
#     for future in as_completed(futures):
#         result = future.result()
#         attachment_records.extend(result)
# 
# import os
# 
# valid_records = [r for r in attachment_records if "error" not in r]
# imgurl_df = pd.DataFrame(valid_records)
# 
# imgurl_df = imgurl_df.merge(
#     ef4_5_df[["objectid", "efscale"]],
#     on="objectid",
#     how="left"
# )
# 
# imgurl_df["image_url"] = imgurl_df.apply(
#     lambda row: f"https://services.dat.noaa.gov/arcgis/rest/services/nws_damageassessmenttoolkit/DamageViewer/FeatureServer/0/{row.objectid}/attachments/{row.attachment_id}",
#     axis=1
# )
# 
# os.makedirs("EF4_EF5_Images", exist_ok=True)
# imgurl_df["download_status"] = "pending"
# 
# # Download with status tracking
# for i, row in imgurl_df.iterrows():
#     try:
#         url = row["image_url"]
#         filename = f'{row["objectid"]}_{row["attachment_id"]}_{row["name"]}'
#         path = os.path.join("EF4_EF5_Images", filename)
# 
#         r = requests.get(url, timeout=10)
#         if r.status_code == 200:
#             with open(path, "wb") as f:
#                 f.write(r.content)
#             imgurl_df.at[i, "download_status"] = "success"
#         else:
#             imgurl_df.at[i, "download_status"] = f"HTTP {r.status_code}"
#     except Exception as e:
#         imgurl_df.at[i, "download_status"] = f"error: {str(e)}"
# 
# # Save log-enhanced CSV
# imgurl_df.to_csv("ef4_ef5_image_log.csv", index=False)


In [ ]:
damage_points

Damage Points:

objectid : 
event_id :
path_guid :
globalid :
device_id :

stormdate :
surveydate :
edit_time :

damage : 
damage_txt :
dod_txt :
efscale :
dod :
windspeed :
qc :

injuries :
deaths :

lat :
lon :
gps_horiz_accuracy :
geometry :

office :
surveytype :
edit_user :
image :

comments :


In [ ]:
damage_points.columns

In [ ]:
damage_lines = gpd.read_file('damage/damage_lines.gpkg')

In [ ]:
damage_lines.columns

In [ ]:
damage_lines

In [ ]:
damage_lines["stormdate"] = pd.to_datetime(damage_lines["stormdate"], unit="ms")
damage_lines["starttime"] = pd.to_datetime(damage_lines["starttime"], unit="ms")
damage_lines["endtime"] = pd.to_datetime(damage_lines["endtime"], unit="ms")
damage_lines["last_edited_date"] = pd.to_datetime(damage_lines["last_edited_date"], unit="ms")

damage_lines

In [ ]:
damage_polygons = gpd.read_file('damage/damage_polygons.gpkg')

In [ ]:
damage_polygons["stormdate"] = pd.to_datetime(damage_polygons["stormdate"], unit="ms")
damage_polygons["created_date"] = pd.to_datetime(damage_polygons["created_date"], unit="ms")
damage_polygons["last_edited_date"] = pd.to_datetime(damage_polygons["last_edited_date"], unit="ms")

In [ ]:
damage_polygons

In [ ]:
import nbformat

def repair_notebook(path_in, path_out=None):
    nb = nbformat.read(path_in, as_version=4)

    for i, cell in enumerate(nb.cells):
        cell.setdefault("metadata", {})

        if cell.cell_type == "code":
            cell.setdefault("execution_count", None)
            cell.setdefault("outputs", [])
            cell.setdefault("source", "")

        elif cell.cell_type == "markdown":
            cell.setdefault("source", "")

    nb.metadata.setdefault("language_info", {})
    nb.metadata.setdefault("kernelspec", {})
    
    path_out = path_out or path_in
    nbformat.write(nb, path_out)

repair_notebook("damage_assessment.ipynb")
